# HW4 — Aggregator inflation on the $C_6$ vs $2K_3$ blind spot

**Reading.** [`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) §4 (E03 ledger, aggregator triple) and **Corollary 3.4** (PA-MPC sandwich for admissible families).

**Goal.** Show that no permutation-invariant node-local aggregator (sum, mean, max) separates $C_6$ from $2K_3$ under constant or degree features; triangle counts globally distinguish them but remain constant *within* each graph. **Q5 (new in r2.1)** computes the aggregator inflation $r_T \in \{\Delta_{\max}, 1, 1\}$ for $T \in \{\text{sum, mean, sym}\}$ — the E03 punchline that quantifies *how much looser* the bracket becomes per aggregator choice.

**Julia companion (optional).** [`julia-theory/notebooks/11_aggregator_triple.jl`](../../../julia-theory/notebooks/11_aggregator_triple.jl) is the reactive-slider twin of Q5; on Cora ($\Delta_{\max}=168$) sum is **7 orders of magnitude** looser than mean.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw4', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
def edges_C6():
    pairs = [(i,(i+1)%6) for i in range(6)]
    e = []
    for u,v in pairs:
        e += [(u,v),(v,u)]
    return np.array(e).T

def edges_2K3():
    pairs = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5)]
    e = []
    for u,v in pairs:
        e += [(u,v),(v,u)]
    return np.array(e).T


## Q1 — Three aggregator partitions.

`sum/mean/max_partition(edge_index, n, x)` groups nodes by `(x_u, σ_u)` where $σ_u$ aggregates over $\{x_v : v \sim u\}$.

In [ ]:
def _neighbours(edge_index, n):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — partition vs label.** The aggregator groups *nodes*; it does not assign class labels. Partition cells coarsen as the key set shrinks.

In [ ]:
# K2 with x=[1,1]: each aggregator → one cell.
ei_k2 = np.array([[0,1],[1,0]])
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    p = fn(ei_k2, 2, np.array([1.0,1.0]))
    print(f'  K2 / {name}: {len(p)} cell(s)')


**Gate Q1.** All three aggregators yield 1 cell on $K_2$ with $x=(1,1)$.

In [ ]:
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    p = fn(ei_k2, 2, np.array([1.0,1.0]))
    assert len(p) == 1, f'Q1: {name} should collapse K2 to 1 cell'
print('[GATE OK] Q1: sum/mean/max collapse K2 with constant features to 1 cell')


In [ ]:
reflect.log('Q4.Q1_aggregators', 'sum/mean/max partitions implemented; collapse on K2/constant', 'HIGH')


## Q2 — Constant features collapse both $C_6$ and $2K_3$ under every aggregator.

In [ ]:
x_const = np.ones(6)
ei_c6, ei_k3 = edges_C6(), edges_2K3()
results = {}
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    for g, ei in [('C6', ei_c6), ('2K3', ei_k3)]:
        p = fn(ei, 6, x_const)
        results[(name,g)] = (len(p), tuple(sorted(len(c) for c in p)))
        print(f'  {g} / {name}: {len(p)} cell(s), sizes {results[(name,g)][1]}')


**Gate Q2.** Every (aggregator × graph) pair returns exactly 1 cell of size 6.

In [ ]:
for k, (ncells, sizes) in results.items():
    assert ncells == 1 and sizes == (6,), f'Q2: {k} should be a single 6-cell, got {ncells},{sizes}'
print('[GATE OK] Q2: constant features → 1×6 on both C6 and 2K3 under sum/mean/max')


In [ ]:
reflect.log('Q4.Q2_constant_collapse', 'all 6 (agg×graph) cases collapse to 1×6', 'HIGH')


## Q3 — Degree feature. Both graphs are 2-regular, so feature is constant ⇒ still blind.

In [ ]:
def degree(edge_index, n):
    deg = np.zeros(n)
    for u in edge_index[0]:
        deg[int(u)] += 1
    return deg

deg_c6, deg_k3 = degree(ei_c6, 6), degree(ei_k3, 6)
print(f'deg(C6)={deg_c6}, deg(2K3)={deg_k3}')


**Gate Q3.** Both degree vectors equal $(2,2,2,2,2,2)$, so the result is identical to Q2.

In [ ]:
assert np.all(deg_c6 == 2) and np.all(deg_k3 == 2), f'Q3: not 2-regular: {deg_c6}, {deg_k3}'
# As consequence, sum/mean/max with degree feature also collapse to 1×6.
for name, fn in [('sum',sum_partition),('mean',mean_partition),('max',max_partition)]:
    for g, ei, deg in [('C6', ei_c6, deg_c6), ('2K3', ei_k3, deg_k3)]:
        p = fn(ei, 6, deg)
        assert len(p) == 1 and len(p[0]) == 6, f'Q3: {g}/{name} did not collapse with degree feature'
print('[GATE OK] Q3: degree feature is constant on both → identical aggregator collapse')


In [ ]:
reflect.log('Q4.Q3_degree_blind', 'degree feature equal to constant on 2-regular graphs', 'HIGH')


## Q4 — Triangle count separates the two graphs globally but is constant within each.

In [ ]:
def triangle_count(edge_index, n):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Triangle count is a *global* invariant that distinguishes graphs but is constant within each graph here → still no within-graph partition refinement.

In [ ]:
t_c6 = triangle_count(ei_c6, 6)
t_k3 = triangle_count(ei_k3, 6)
print(f'triangles(C6)={t_c6}  sum={t_c6.sum()}')
print(f'triangles(2K3)={t_k3}  sum={t_k3.sum()}')


**Gate Q4.** $C_6$ → all 0; $2K_3$ → all 1; graphs distinguished by feature sum.

In [ ]:
assert np.all(t_c6 == 0), f'Q4: C6 triangle vector {t_c6} ≠ 0'
assert np.all(t_k3 == 1), f'Q4: 2K3 triangle vector {t_k3} ≠ 1'
assert t_c6.sum() != t_k3.sum(), 'Q4: triangle feature did not separate the two graphs'
print('[GATE OK] Q4: triangles(C6)=0 everywhere, triangles(2K3)=1 everywhere')


In [ ]:
reflect.log('Q4.Q4_triangles', 'triangle-count separates C6 from 2K3 globally; constant within each', 'HIGH')


## Q5 — Aggregator inflation $r_T$ and the bracket-looseness budget

**Paper §4 / E03 punchline.** Each aggregator $T$ inflates the Theorem-1 upper envelope by a constant $r_T(\Delta_{\max})$ depending on the maximum graph degree:

| aggregator $T$ | $r_T(\Delta_{\max})$ | rationale |
|---|---|---|
| **sum**  | $\Delta_{\max}$ | output magnitude grows with $\deg$ |
| **mean** | $1$              | normalised by $\deg$ |
| **sym** (GCN) | $1$        | symmetric $D^{-1/2}AD^{-1/2}$ normalisation |

On Cora ($\Delta_{\max}=168$) the sum-aggregator upper envelope is **two-and-a-half orders of magnitude** looser than mean's. This is **honest looseness, not a bug**: the inequality still holds, it's just trivially satisfied.

**Bridge to Cor 3.4.** Corollary 3.4(2) gives $\varepsilon^{\text{model}} \le r_T \cdot \tfrac{1}{2} H(Y\mid\Pi_\mathcal{A})$ for any admissible family $\mathcal{A}$ (Def 3.5). The number $r_T$ is *not* a property of the partition; it's a property of the aggregator. Choose mean/sym to get a tight architectural bracket; choose sum and accept that your upper bound is loose.

In [ ]:
def r_T(aggregator: str, delta_max: int) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — `r_T` vs the bracket.** The bracket itself is $[H_\mathrm{bin}^{-1}(h),\, h/2]$ on any $\Pi$; $r_T$ rescales **only the upper side** when the partition is *architecture-induced* (Cor 3.4). The lower side $H_\mathrm{bin}^{-1}(h)$ has no aggregator scaling — Fano is information-theoretic, not architectural.

In [ ]:
# Compute Δmax for the two toy graphs and Cora-realistic regime.
def delta_max(edge_index, n):
    d = np.zeros(n, dtype=int)
    for u in edge_index[0]:
        d[int(u)] += 1
    return int(d.max())

dm_c6   = delta_max(ei_c6, 6)
dm_k3   = delta_max(ei_k3, 6)
dm_cora = 168  # paper value for Cora's max degree
print(f'Δmax(C6)={dm_c6}  Δmax(2K3)={dm_k3}  Δmax(Cora)={dm_cora}')

rows = []
for T in ('sum', 'mean', 'sym'):
    rows.append((T, r_T(T, dm_c6), r_T(T, dm_k3), r_T(T, dm_cora)))
print(f"{'aggregator':<10}  {'r_T(C6)':>10}  {'r_T(2K3)':>10}  {'r_T(Cora)':>10}")
for T, a, b, c in rows:
    print(f'{T:<10}  {a:>10.1f}  {b:>10.1f}  {c:>10.1f}')

ratio = r_T('sum', dm_cora) / r_T('mean', dm_cora)
print(f'\non Cora: sum upper is {ratio:.0f}× looser than mean upper')


**Gate Q5.** $r_\text{mean}=r_\text{sym}=1$; $r_\text{sum}=\Delta_{\max}$; sum-vs-mean ratio on Cora is 168.

In [ ]:
assert r_T('mean', 1)   == 1.0 and r_T('mean', 9999) == 1.0, 'Q5: r_mean must be 1 for any Δmax'
assert r_T('sym', 1)    == 1.0 and r_T('sym', 9999)  == 1.0, 'Q5: r_sym must be 1 for any Δmax'
assert r_T('sum', 168) == 168.0, f'Q5: r_sum(168)={r_T("sum", 168)} ≠ 168'
assert ratio == 168.0, f'Q5: Cora sum/mean ratio {ratio} ≠ 168'
# Bracket sanity: r_T·upper(h) ≥ upper(h) (looseness is monotone in r_T).
from math import log2
_hbin = lambda p: 0.0 if (p <= 0 or p >= 1) else -(p*log2(p) + (1-p)*log2(1-p))
for h in (0.1, 0.5, 0.7219, 0.9):
    assert r_T('sum', 168) * 0.5 * h >= 0.5 * h, 'Q5: r_sum-inflated upper must dominate plain upper'
print(f'[GATE OK] Q5: r_mean=r_sym=1, r_sum=Δmax; on Cora sum is {int(ratio)}× looser than mean')


In [ ]:
reflect.log('Q4.Q5_aggregator_inflation',
            f'r_T computed for sum/mean/sym; on Cora sum is {int(ratio)}× looser than mean (honest looseness)',
            'HIGH')


## Q6 — Writeup & calibration.

In [ ]:
reflect.log('Q4.Q6_writeup', 'no permutation-invariant node-local aggregator separates C6 from 2K3; r_T inflation tabulated', 'HIGH')
print('HW4 reflection log:')
for r in reflect.dump():
    if 'hw4' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW4.**